# Traffix.id — Inference Pipeline
YOLO Detection → Traffic Feature Extraction → LSTM 15-Minute Prediction

## 1. Install Dependencies

In [ ]:
!pip install ultralytics tensorflow opencv-python-headless joblib requests -q
!pip install tf-keras -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.0 MB/s eta 0:00:00


## 2. Imports

In [ ]:
import os, gc, re, json, warnings
import numpy as np
import pandas as pd
import cv2
import joblib
import requests
import tensorflow as tf
from collections import defaultdict
from datetime import datetime

warnings.filterwarnings('ignore')

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

print("Imports OK.")

Imports OK.


## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Upload Model Artifacts
`best.pt`, `best_lstm_15m.keras`, `feat_scaler.joblib`, `target_scalers.joblib`, `best_config.json`, `feature_columns.json`, `video_weather_data.csv`

In [ ]:
from google.colab import files as colab_files

ARTIFACTS_DIR = '/content/artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

uploaded = colab_files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(ARTIFACTS_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"Saved: {dest}")

Saving video_weather_data.csv to video_weather_data.csv
Saving best.pt to best.pt
Saving best_config.json to best_config.json
Saving best_lstm_15m.keras to best_lstm_15m.keras
Saving feat_scaler.joblib to feat_scaler.joblib
Saving target_scalers.joblib to target_scalers.joblib
Saving feature_columns.json to feature_columns.json
Saved: /content/artifacts/video_weather_data.csv
Saved: /content/artifacts/best.pt
Saved: /content/artifacts/best_config.json
Saved: /content/artifacts/best_lstm_15m.keras
Saved: /content/artifacts/feat_scaler.joblib
Saved: /content/artifacts/target_scalers.joblib
Saved: /content/artifacts/feature_columns.json


## 5. Runtime Configuration

In [ ]:
# Demo / performance controls
DEMO_MODE             = False # True = process fewer videos; False = all videos
MAX_VIDEOS_PER_PERIOD = 2    # max videos per folder when DEMO_MODE=True
MAX_FRAMES_PER_VIDEO  = 300  # hard cap on frames processed per video
FRAME_STRIDE          = 5    # process every Nth frame (1 = all frames)
YOLO_IMGSZ            = 416  # inference image size (416 or 512 for speed)
YOLO_CONF             = 0.20 # detection confidence threshold (lowered for dense traffic)
YOLO_IOU              = 0.60 # IOU threshold for NMS (higher = keep more overlapping boxes)
YOLO_MAX_DET          = 500  # max detections per frame
SAVE_DETECTED_VIDEO   = True # set False to skip writing output video
SAVE_EVERY_N_FRAMES   = 2    # write only every Nth frame to output video

# Output paths
OUTPUT_DIR   = '/content/outputs'
DETECTED_DIR = os.path.join(OUTPUT_DIR, 'detected_videos')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DETECTED_DIR, exist_ok=True)

# Video source
PERIODS          = ['pagi', 'siang', 'malam']
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

print(f"DEMO_MODE={DEMO_MODE} | MAX_FRAMES={MAX_FRAMES_PER_VIDEO} | "
      f"STRIDE={FRAME_STRIDE} | IMGSZ={YOLO_IMGSZ} | CONF={YOLO_CONF} | "
      f"IOU={YOLO_IOU} | MAX_DET={YOLO_MAX_DET}")
print("Output directories ready.")

DEMO_MODE=False | MAX_FRAMES=300 | STRIDE=5 | IMGSZ=416 | CONF=0.2 | IOU=0.6 | MAX_DET=500
Output directories ready.


## 6. Load LSTM Artifacts

In [ ]:
ARTIFACTS_DIR = '/content/artifacts'

with open(os.path.join(ARTIFACTS_DIR, 'best_config.json')) as f:
    config = json.load(f)

SEQ_LEN    = config['seq_len']
HORIZON    = config['horizon']
MODEL_MAPE = config['test_mape_15m']
print(f"Config: seq_len={SEQ_LEN} | horizon={HORIZON} | MAPE={MODEL_MAPE}")

with open(os.path.join(ARTIFACTS_DIR, 'feature_columns.json')) as f:
    FEATURE_COLUMNS = json.load(f)
NUM_FEATURES = len(FEATURE_COLUMNS)
print(f"Features: {NUM_FEATURES} columns")

feat_scaler    = joblib.load(os.path.join(ARTIFACTS_DIR, 'feat_scaler.joblib'))
target_scalers = joblib.load(os.path.join(ARTIFACTS_DIR, 'target_scalers.joblib'))
print("Scalers loaded.")

lstm_model = tf.keras.models.load_model(
    os.path.join(ARTIFACTS_DIR, 'best_lstm_15m.keras'), compile=False)
print("LSTM model loaded.")
lstm_model.summary()

Config: seq_len=60 | horizon=15m | MAPE=20.6046
Features: 41 columns
Scalers loaded.
LSTM model loaded.


Model: "lstm_light"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 60, 41)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm1 (LSTM)                    │ (None, 60, 64)         │        27,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm2 (LSTM)                    │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,729 (163.00 KB)

 Trainable params: 41,729 (163.00 KB)

 Non-trainable params: 0 (0.00 B)

## 7. Load YOLO Model (once)

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO(os.path.join(ARTIFACTS_DIR, 'best.pt'))
print("Model class names:")
for cid, cname in yolo_model.names.items():
    print(f"  {cid}: {cname}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model class names:
  0: car


## 8. Load Video Metadata CSV

In [ ]:
CSV_PATH = os.path.join(ARTIFACTS_DIR, 'video_weather_data.csv')
meta_df  = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

print(f"Columns  : {list(meta_df.columns)}")
print(f"Rows     : {len(meta_df)}")
print(f"Duplicates on video_file: {meta_df['video_file'].duplicated().sum()}")
print(f"Missing overlay_date    : {meta_df['overlay_date'].isna().sum()}")
print(f"Missing overlay_time    : {meta_df['overlay_time'].isna().sum()}")
print(f"Missing lat             : {meta_df['lat'].isna().sum()}")
print(f"Missing lon             : {meta_df['lon'].isna().sum()}")
print(f"Missing temperature_2m  : {meta_df['temperature_2m'].isna().sum()}")
print(f"Missing weather_status  : {meta_df['weather_status'].isna().sum()}")

# Build lookup dict: video_file -> row dict
meta_lookup = {row['video_file']: row.to_dict() for _, row in meta_df.iterrows()}
print(f"\nMetadata lookup built: {len(meta_lookup)} entries.")

Columns  : ['no', 'period', 'period_from_filename', 'cctv_id', 'overlay_date', 'overlay_time', 'timestamp_wib', 'ruas', 'video_file', 'selected_frame_path', 'selected_frame_second', 'selected_frame_file', 'location_type', 'location_text', 'km_text', 'km_decimal', 'manual_status', 'filename_parse_status', 'notes', 'camera_label', 'camera_key', 'camera_id', 'lat', 'lon', 'coordinate_source', 'confidence', 'coordinate_notes', 'coordinate_status', 'coordinate_issues', 'join_coordinate_status', 'weather_hour', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m', 'weather_code', 'weather_status', 'weather_error']
Rows     : 21
Duplicates on video_file: 0
Missing overlay_date    : 0
Missing overlay_time    : 0
Missing lat             : 0
Missing lon             : 0
Missing temperature_2m  : 0
Missing weather_status  : 0

Metadata lookup built: 21 entries.


## 9. Weather & Metadata Helpers

In [ ]:
OWM_API_KEY = ''   # enable live API fallback / ignore

WEATHER_CODE_MAP = {
    0: 'Clear', 1: 'Clear', 2: 'Cloudy', 3: 'Cloudy',
    45: 'Cloudy', 48: 'Cloudy',
    51: 'Rain', 53: 'Rain', 55: 'Rain',
    61: 'Rain', 63: 'Rain', 65: 'Rain',
    80: 'Rain', 81: 'Rain', 82: 'Rain',
    95: 'Rain', 96: 'Rain', 99: 'Rain',
}

def normalize_weather_label(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return 'Clear'
    s = str(raw).strip().lower()
    if any(k in s for k in ['clear', 'sunny', 'cerah', 'fine']):
        return 'Clear'
    if any(k in s for k in ['cloud', 'overcast', 'berawan', 'mendung']):
        return 'Cloudy'
    if any(k in s for k in ['hot', 'panas', 'terik']):
        return 'Hot'
    if any(k in s for k in ['rain', 'drizzle', 'hujan', 'shower', 'thunder']):
        return 'Rain'
    return 'Clear'

def get_weather_from_api(lat, lon, api_key=None):
    if not api_key:
        return None
    try:
        url  = (f"https://api.openweathermap.org/data/2.5/weather"
                f"?lat={lat}&lon={lon}&appid={api_key}&units=metric")
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        data   = resp.json()
        desc   = data['weather'][0]['main']
        temp_c = round(data['main']['temp'], 1)
        label  = normalize_weather_label(desc)
        if temp_c >= 36:
            label = 'Hot'
        return {'weather': label, 'weather_temp_c': temp_c, 'source': 'openweathermap'}
    except Exception as e:
        print(f"[WARN] Weather API failed: {e}")
        return None

def extract_metadata_for_video(video_filename):
    basename = os.path.basename(video_filename)
    row      = meta_lookup.get(basename)

    # Date & time
    date_str, time_str, hour = None, None, None

    if row:
        raw_date = str(row.get('overlay_date', '') or '').strip()
        raw_time = str(row.get('overlay_time', '') or '').strip()
        if raw_date and raw_date != 'nan':
            date_str = raw_date
        if raw_time and raw_time != 'nan':
            time_str = raw_time
            try:
                hour = int(raw_time.split(':')[0])
            except:
                pass

    if not date_str or not time_str:
        # Fallback: parse from filename
        m = re.search(r'(\d{8})[_\-](\d{6})', basename)
        if m:
            try:
                dt       = datetime.strptime(m.group(1) + m.group(2), '%Y%m%d%H%M%S')
                date_str = date_str or dt.strftime('%Y-%m-%d')
                time_str = time_str or dt.strftime('%H:%M:%S')
                hour     = hour or dt.hour
            except:
                pass

    if not date_str:
        now      = datetime.now()
        date_str = now.strftime('%Y-%m-%d')
        time_str = time_str or now.strftime('%H:%M:%S')
        hour     = hour or now.hour

    # Coordinates
    lat, lon = None, None
    if row:
        for col in ['lat', 'latitude']:
            if col in row and pd.notna(row[col]):
                lat = float(row[col]); break
        for col in ['lon', 'lng', 'longitude']:
            if col in row and pd.notna(row[col]):
                lon = float(row[col]); break

    # Weather
    weather, temp_c = 'Clear', 32.0
    weather_source  = 'default'

    if row:
        # Temperature:
        if 'temperature_2m' in row and pd.notna(row.get('temperature_2m')):
            try:
                temp_c = float(row['temperature_2m'])
            except:
                pass

        # Weather label: try weather_code first, then text columns
        wcode = row.get('weather_code')
        if pd.notna(wcode) and int(wcode) in WEATHER_CODE_MAP:
            weather = WEATHER_CODE_MAP[int(wcode)]
            if temp_c >= 36:
                weather = 'Hot'
            weather_source = 'csv_weather_code'
        elif 'weather_status' in row and pd.notna(row.get('weather_status')):
            weather = normalize_weather_label(str(row['weather_status']))
            weather_source = 'csv_weather_status'

    # Camera & location
    camera_id     = row.get('camera_id', '') if row else ''
    location_text = row.get('location_text', '') if row else ''
    ruas          = row.get('ruas', 'Jakarta-Tangerang') if row else 'Jakarta-Tangerang'

    dt_obj = datetime.now()
    try:
        dt_obj = datetime.strptime(f"{date_str} {time_str}", '%Y-%m-%d %H:%M:%S')
    except:
        pass

    return {
        'date':          date_str,
        'time':          time_str,
        'hour':          hour or 0,
        'minute':        dt_obj.minute,
        'day':           dt_obj.day,
        'day_of_week':   dt_obj.weekday(),
        'month':         dt_obj.month,
        'is_weekend':    1 if dt_obj.weekday() >= 5 else 0,
        'lat':           lat,
        'lon':           lon,
        'weather':       weather,
        'weather_temp_c': round(temp_c, 1),
        'weather_source': weather_source,
        'camera_id':     camera_id,
        'location_text': location_text,
        'ruas':          ruas,
        'metadata_found': row is not None,
    }

def get_period_label(hour):
    if  5 <= hour <  9: return 'Morning Rush'
    elif 9 <= hour < 16: return 'Daytime'
    elif 16 <= hour < 19: return 'Evening Rush'
    elif 19 <= hour < 23: return 'Night'
    else:                 return 'Late Night'

print("Weather and metadata helpers ready.")

Weather and metadata helpers ready.


## 10. Scan Video Folders from Google Drive (Shared with Me)

In [ ]:
def find_shared_data_root():
    candidates = []
    shared_drives = '/content/drive/Shareddrives'
    if os.path.isdir(shared_drives):
        for d in os.listdir(shared_drives):
            p = os.path.join(shared_drives, d, 'video_cctv')
            if os.path.isdir(p):
                candidates.append(p)
    my_drive = '/content/drive/MyDrive'
    if os.path.isdir(my_drive):
        for item in os.listdir(my_drive):
            p = os.path.join(my_drive, item, 'video_cctv')
            if os.path.isdir(p):
                candidates.append(p)
        p = os.path.join(my_drive, 'video_cctv')
        if os.path.isdir(p):
            candidates.append(p)
    return candidates

candidates = find_shared_data_root()
if candidates:
    VIDEO_ROOT = candidates[0]
else:
    VIDEO_ROOT = '/content/drive/MyDrive/video_cctv'
    print(f"[WARN] Fallback: {VIDEO_ROOT}")
print(f"Video root: {VIDEO_ROOT}")

def scan_videos(root, periods):
    summary = {}
    for period in periods:
        folder = os.path.join(root, period)
        if not os.path.isdir(folder):
            print(f"[WARN] Not found: {folder}")
            summary[period] = []
            continue
        files = []
        for f in sorted(os.listdir(folder)):
            if os.path.splitext(f)[1].lower() in VIDEO_EXTENSIONS:
                fp      = os.path.join(folder, f)
                size_mb = os.path.getsize(fp) / 1024 / 1024
                files.append({'name': f, 'path': fp, 'size_mb': round(size_mb, 2)})
        summary[period] = files
    return summary

video_summary = scan_videos(VIDEO_ROOT, PERIODS)
for period, files in video_summary.items():
    print(f"\n--- {period.upper()} ({len(files)} videos) ---")
    for v in files:
        print(f"  {v['name']}  {v['size_mb']} MB")

Video root: /content/drive/MyDrive/video_cctv

--- PAGI (7 videos) ---
  pagi_cctv_001_20260518_070935.mp4  0.26 MB
  pagi_cctv_002_20260518_070954.mp4  3.45 MB
  pagi_cctv_004_20260518_071356.mp4  0.65 MB
  pagi_cctv_006_20260518_071450.mp4  1.93 MB
  pagi_cctv_009_20260518_071625.mp4  0.63 MB
  pagi_cctv_014_20260518_072255.mp4  0.31 MB
  pagi_cctv_015_20260518_072703.mp4  4.91 MB

--- SIANG (6 videos) ---
  siang_cctv_001_20260518_163906.mp4  7.72 MB
  siang_cctv_003_20260518_165921.mp4  11.69 MB
  siang_cctv_004_20260518_170659.mp4  5.64 MB
  siang_cctv_005_20260518_171425.mp4  10.79 MB
  siang_cctv_007_20260518_160452.mp4  9.8 MB
  siang_cctv_008_20260518_151934.mp4  0.43 MB

--- MALAM (8 videos) ---
  malam_cctv_001_20260518_003725.mp4  3.47 MB
  malam_cctv_003_20260518_004022.mp4  1.69 MB
  malam_cctv_005_20260518_004357.mp4  2.03 MB
  malam_cctv_007_20260518_033146.mp4  0.99 MB
  malam_cctv_008_20260518_004504.mp4  0.51 MB
  malam_cctv_009_20260518_004524.mp4  0.48 MB
  malam_c

## 11. Select Videos for Processing

In [ ]:
selected_videos = []
for period, files in video_summary.items():
    limit = MAX_VIDEOS_PER_PERIOD if DEMO_MODE else len(files)
    for v in files[:limit]:
        v['period_label'] = period
        selected_videos.append(v)

print(f"Selected {len(selected_videos)} video(s) (DEMO_MODE={DEMO_MODE}):")
for v in selected_videos:
    meta = extract_metadata_for_video(v['name'])
    v['meta'] = meta
    found_str = 'CSV match' if meta['metadata_found'] else 'no CSV match'
    print(f"  [{v['period_label']}] {v['name']}  "
          f"date={meta['date']} time={meta['time']}  "
          f"weather={meta['weather']} {meta['weather_temp_c']}C  "
          f"({found_str})")

Selected 21 video(s) (DEMO_MODE=False):
  [pagi] pagi_cctv_001_20260518_070935.mp4  date=5/18/2026 time=6:09:33  weather=Cloudy 25.6C  (CSV match)
  [pagi] pagi_cctv_002_20260518_070954.mp4  date=5/18/2026 time=6:15:26  weather=Cloudy 26.8C  (CSV match)
  [pagi] pagi_cctv_004_20260518_071356.mp4  date=5/18/2026 time=6:18:01  weather=Cloudy 25.6C  (CSV match)
  [pagi] pagi_cctv_006_20260518_071450.mp4  date=5/18/2026 time=6:20:24  weather=Cloudy 25.2C  (CSV match)
  [pagi] pagi_cctv_009_20260518_071625.mp4  date=5/18/2026 time=6:20:49  weather=Cloudy 25.6C  (CSV match)
  [pagi] pagi_cctv_014_20260518_072255.mp4  date=5/18/2026 time=6:23:39  weather=Cloudy 25.2C  (CSV match)
  [pagi] pagi_cctv_015_20260518_072703.mp4  date=5/18/2026 time=6:39:21  weather=Cloudy 25.2C  (CSV match)
  [siang] siang_cctv_001_20260518_163906.mp4  date=5/18/2026 time=15:49:02  weather=Rain 29.9C  (CSV match)
  [siang] siang_cctv_003_20260518_165921.mp4  date=5/18/2026 time=16:17:21  weather=Clear 29.7C  (CSV m

## 12. Vehicle Class IDs & Frame Preprocessing

In [ ]:
VEHICLE_CLASS_NAMES = {"car", "motorcycle", "bus", "truck"}

def get_vehicle_class_ids(model):
    """Whitelist-based: only detect actual vehicle classes."""
    ids = [cid for cid, cname in model.names.items()
           if cname.lower() in VEHICLE_CLASS_NAMES]
    if not ids:
        # Fallback: exclude known non-vehicle keywords
        non_veh = {'person', 'pedestrian', 'people', 'human', 'rider',
                   'bicycle', 'backpack', 'handbag', 'umbrella'}
        ids = [cid for cid, cname in model.names.items()
               if cname.lower() not in non_veh]
    return set(ids)

VEHICLE_CLASS_IDS = get_vehicle_class_ids(yolo_model)
print(f"Vehicle class IDs  : {sorted(VEHICLE_CLASS_IDS)}")
print(f"Vehicle class names: {[yolo_model.names[i] for i in sorted(VEHICLE_CLASS_IDS)]}")
print(f"Excluded classes   : {[n for i,n in yolo_model.names.items() if i not in VEHICLE_CLASS_IDS]}")

ZONE_NAMES = ['LEFT', 'CENTER', 'RIGHT']

def get_zone(cx, frame_width):
    t = frame_width / 3
    if cx < t:     return 'LEFT'
    elif cx < 2*t: return 'CENTER'
    else:          return 'RIGHT'

def draw_zones(frame):
    h, w = frame.shape[:2]
    t = w // 3
    cv2.line(frame, (t, 0), (t, h), (0, 255, 255), 1)
    cv2.line(frame, (2*t, 0), (2*t, h), (0, 255, 255), 1)
    for i, name in enumerate(ZONE_NAMES):
        cv2.putText(frame, name, (int((i+0.5)*t) - 25, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    return frame

def preprocess_frame(frame):
    """
    Auto preprocessing: detect mean brightness.
    Dark  (mean < 60): CLAHE + brightness boost.
    Normal           : sharpening + slight contrast.
    """
    mean_b = float(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).mean())
    if mean_b < 60:
        lab      = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l, a, b  = cv2.split(lab)
        l_eq     = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)
        frame    = cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)
        frame    = cv2.convertScaleAbs(frame, alpha=1.3, beta=15)
    else:
        kernel = np.array([[0,-0.5,0],[-0.5,3,-0.5],[0,-0.5,0]])
        frame  = cv2.filter2D(frame, -1, kernel)
        frame  = cv2.convertScaleAbs(frame, alpha=1.05, beta=0)
    return frame

print("Preprocessing helpers ready.")

Vehicle class IDs  : [0]
Vehicle class names: ['car']
Excluded classes   : []
Preprocessing helpers ready.


## 13. Speed Estimation

In [ ]:
import math
from collections import defaultdict

PX_PER_METER          = 8.0    # rough calibration; tune per camera if known
CENTROID_MAX_DIST     = 80     # pixels — max distance to match a detection to existing track
TRACK_MAX_MISSING     = 10     # frames a track can be absent before being pruned
STATIONARY_THRESHOLD  = 5     # pixels — movement below this = stationary
STATIONARY_MIN_FRAMES = 10    # consecutive frames to confirm stationary

def centroid(x1, y1, x2, y2):
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

def euclidean(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

class CentroidTracker:
    def __init__(self, max_dist=CENTROID_MAX_DIST, max_missing=TRACK_MAX_MISSING):
        self.next_id         = 0
        self.tracks          = {}   # tid -> {centroid, last_seen, box_history, stationary_frames}
        self.unique_count    = 0
        self.max_dist        = max_dist
        self.max_missing     = max_missing
        self.frame_num       = 0

    def update(self, detections):
        self.frame_num += 1
        new_centroids = [centroid(x1,y1,x2,y2) for (x1,y1,x2,y2) in detections]

        track_ids      = list(self.tracks.keys())
        matched_tracks = set()
        matched_dets   = set()
        pairs          = []

        for tid in track_ids:
            tc = self.tracks[tid]['centroid']
            for di, dc in enumerate(new_centroids):
                d = euclidean(tc, dc)
                if d < self.max_dist:
                    pairs.append((d, tid, di))

        pairs.sort()
        matched = []
        for dist, tid, di in pairs:
            if tid in matched_tracks or di in matched_dets:
                continue
            matched_tracks.add(tid)
            matched_dets.add(di)
            x1,y1,x2,y2 = detections[di]
            prev_centroid = self.tracks[tid]['centroid']
            new_c = new_centroids[di]
            movement = euclidean(prev_centroid, new_c)

            # Update stationary frame counter
            if movement < STATIONARY_THRESHOLD:
                self.tracks[tid]['stationary_frames'] = (
                    self.tracks[tid].get('stationary_frames', 0) + 1)
            else:
                self.tracks[tid]['stationary_frames'] = 0

            self.tracks[tid]['centroid']    = new_c
            self.tracks[tid]['last_seen']   = self.frame_num
            self.tracks[tid]['box_history'].append((x1,y1,x2,y2))
            if len(self.tracks[tid]['box_history']) > 20:
                self.tracks[tid]['box_history'].pop(0)
            matched.append((tid, x1, y1, x2, y2))

        for di, (x1,y1,x2,y2) in enumerate(detections):
            if di in matched_dets:
                continue
            tid = self.next_id
            self.next_id      += 1
            self.unique_count += 1
            self.tracks[tid] = {
                'centroid':           new_centroids[di],
                'last_seen':          self.frame_num,
                'box_history':        [(x1,y1,x2,y2)],
                'stationary_frames':  0,
            }
            matched.append((tid, x1, y1, x2, y2))

        # Prune stale tracks
        stale = [tid for tid, t in self.tracks.items()
                 if self.frame_num - t['last_seen'] > self.max_missing]
        for tid in stale:
            del self.tracks[tid]

        return matched, len([d for d in range(len(detections)) if d not in matched_dets])

    def get_speed_for_track(self, tid, effective_fps):
        """Estimate speed (km/h) from centroid displacement of track tid."""
        if tid not in self.tracks:
            return None
        hist = self.tracks[tid]['box_history']
        if len(hist) < 2:
            return None
        x1a,y1a,x2a,y2a = hist[-2]
        x1b,y1b,x2b,y2b = hist[-1]
        dist_px = euclidean(centroid(x1a,y1a,x2a,y2a),
                            centroid(x1b,y1b,x2b,y2b))
        kmh = (dist_px / PX_PER_METER) * effective_fps * 3.6
        return round(kmh, 1) if 0 < kmh < 150 else None

    def get_stationary_count(self):
        """Count tracks confirmed as stationary (slow for >= STATIONARY_MIN_FRAMES)."""
        return sum(1 for t in self.tracks.values()
                   if t.get('stationary_frames', 0) >= STATIONARY_MIN_FRAMES)


def fallback_speed(avg_occ, capacity=15.0):
    """Occupancy-based speed fallback. Range 8–80 km/h."""
    ratio = min(avg_occ / max(capacity, 1), 1.0)
    return round(max(8.0, 80.0 - ratio * 72.0), 1)

print("CentroidTracker (with stationary tracking) and speed helpers ready.")

CentroidTracker (with stationary tracking) and speed helpers ready.


## 14. YOLO Inference — Per-Video

In [ ]:
def run_yolo_on_video(video_path, video_name, meta):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Cannot open: {video_path}")
        return None

    fps          = cap.get(cv2.CAP_PROP_FPS) or 25.0
    orig_w       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_area   = orig_w * orig_h

    out_path = os.path.join(DETECTED_DIR, f"detected_{video_name}.mp4")
    writer   = None
    if SAVE_DETECTED_VIDEO:
        writer = cv2.VideoWriter(
            out_path,
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps,
            (orig_w, orig_h),
        )

    tracker         = CentroidTracker()
    frame_data      = []
    yolo_count      = 0
    frames_read     = 0
    frames_written  = 0

    last_matched        = []
    last_n_in_frame     = 0
    last_frame_spd      = None
    last_area_ratio     = 0.0
    last_bbox_area      = 0
    last_stationary_cnt = 0

    effective_fps = fps / FRAME_STRIDE

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames_read += 1
        is_yolo_frame = (
            (frames_read - 1) % FRAME_STRIDE == 0 and
            yolo_count < MAX_FRAMES_PER_VIDEO
        )

        if is_yolo_frame:
            yolo_count += 1
            proc = preprocess_frame(frame)

            results = yolo_model.predict(
                proc,
                imgsz       = YOLO_IMGSZ,
                conf        = YOLO_CONF,
                iou         = YOLO_IOU,
                max_det     = YOLO_MAX_DET,
                agnostic_nms= False,
                verbose     = False,
                classes     = list(VEHICLE_CLASS_IDS),
            )

            raw_dets  = []
            bbox_area = 0
            if results[0].boxes is not None:
                for box in results[0].boxes:
                    cid = int(box.cls[0])
                    if cid not in VEHICLE_CLASS_IDS:
                        continue
                    x1,y1,x2,y2 = map(int, box.xyxy[0])
                    raw_dets.append((x1,y1,x2,y2))
                    bbox_area += (x2-x1)*(y2-y1)

            matched, _ = tracker.update(raw_dets)

            frame_speeds = []
            for (tid,x1,y1,x2,y2) in matched:
                spd = tracker.get_speed_for_track(tid, effective_fps)
                if spd is not None:
                    frame_speeds.append(spd)

            last_matched        = matched
            last_n_in_frame     = len(matched)
            last_frame_spd      = round(float(np.mean(frame_speeds)), 1) if frame_speeds else None
            last_area_ratio     = round(bbox_area / max(frame_area, 1), 4)
            last_bbox_area      = bbox_area
            last_stationary_cnt = tracker.get_stationary_count()

            frame_data.append({
                'n_in_frame':      last_n_in_frame,
                'avg_speed':       last_frame_spd,
                'area_ratio':      last_area_ratio,
                'bbox_area':       last_bbox_area,
                'stationary_cnt':  last_stationary_cnt,
            })

            if yolo_count % 50 == 0:
                print(f"  [{video_name}] yolo_frames={yolo_count} | "
                      f"unique_vehicles={tracker.unique_count} | "
                      f"stationary={last_stationary_cnt}", end='\r')

        if SAVE_DETECTED_VIDEO and writer:
            disp = frame.copy()

            for (tid,x1,y1,x2,y2) in last_matched:
                cv2.rectangle(disp, (x1,y1), (x2,y2), (0,200,0), 2)
                cv2.putText(disp, "vehicle", (x1, max(y1-5,0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.40, (0,200,0), 1)

            cumul   = tracker.unique_count
            spd_str = f"{last_frame_spd} km/h" if last_frame_spd else "--"

            label_txt  = f"Vehicle Count: {cumul}   Speed: {spd_str}"
            (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
            pad = 6
            x1_box = orig_w - tw - pad * 2 - 8
            y1_box = orig_h - th - pad * 2 - 8
            x2_box = orig_w - 8
            y2_box = orig_h - 8
            cv2.rectangle(disp, (x1_box, y1_box), (x2_box, y2_box), (0, 0, 0), -1)
            cv2.putText(disp, label_txt,
                        (x1_box + pad, y2_box - pad),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

            writer.write(disp)
            frames_written += 1

    cap.release()
    if writer:
        writer.release()
    gc.collect()
    if TORCH_AVAILABLE:
        try:
            import torch
            torch.cuda.empty_cache()
        except:
            pass

    cumulative_count = tracker.unique_count
    duration_s       = frames_read / fps

    print(f"\n  [{video_name}] DONE")
    print(f"    source_video          : {video_name}")
    print(f"    original_fps          : {fps}")
    print(f"    total_frames_read     : {frames_read}")
    print(f"    total_frames_written  : {frames_written}")
    print(f"    frame_stride          : {FRAME_STRIDE}")
    print(f"    yolo_frames_processed : {yolo_count}")
    print(f"    unique_vehicles       : {cumulative_count}")
    print(f"    duration_s            : {round(duration_s,1)}")
    if SAVE_DETECTED_VIDEO and frames_written > 0:
        ratio = frames_written / max(frames_read, 1)
        ok    = '[OK]     ' if ratio >= 0.80 else '[WARNING]'
        print(f"    {ok} frames_written/frames_read = {round(ratio,3)}")
        print(f"    Saved: {out_path}")

    return {
        'frame_data':        frame_data,
        'fps':               fps,
        'total_frames':      frames_read,
        'yolo_frames':       yolo_count,
        'frames_written':    frames_written,
        'detected_path':     out_path if SAVE_DETECTED_VIDEO else '',
        'video_name':        video_name,
        'cumulative_count':  cumulative_count,
        'duration_s':        duration_s,
        'frame_area':        frame_area,
    }


## 15. Feature Extraction — Per-Video

In [ ]:
OCCUPANCY_CAPACITY = 15.0   # max expected simultaneous vehicles in frame
SLOW_THRESHOLD     = 15.0   # km/h — below this, vehicle is queuing
EXPECTED_CAPACITY  = 60     # expected max vehicles per clip for count_density

def extract_video_features(result, video_meta):
    frame_data       = result['frame_data']
    fps              = result['fps']
    cumulative_count = result['cumulative_count']
    duration_s       = result['duration_s']
    frame_area       = result.get('frame_area', 1280 * 720)

    if len(frame_data) == 0 or duration_s < 0.5:
        print(f"[WARN] Insufficient data for {result['video_name']}")
        return None

    duration_min = duration_s / 60.0
    meta         = video_meta['meta']

    # Flow (integer outputs)
    vehicle_count      = int(round(cumulative_count))
    vehicle_count_1min = int(round(cumulative_count / max(duration_min, 1e-6)))
    volume_veh_per_hour = int(vehicle_count_1min * 60)

    # Occupancy
    occ_vals = [f['n_in_frame'] for f in frame_data]
    avg_occ  = float(np.mean(occ_vals))

    # Speed
    raw_speeds = [f['avg_speed'] for f in frame_data if f['avg_speed'] is not None]
    if raw_speeds:
        avg_speed_kmh = round(float(np.mean(raw_speeds)), 2)
        slow_frames   = sum(1 for s in raw_speeds if s < SLOW_THRESHOLD)
        slow_ratio    = slow_frames / len(raw_speeds)
    else:
        avg_speed_kmh = fallback_speed(avg_occ, OCCUPANCY_CAPACITY)
        slow_ratio    = min(avg_occ / OCCUPANCY_CAPACITY, 1.0)

    # Stationary vehicle count
    stationary_vals = [f.get('stationary_cnt', 0) for f in frame_data]
    stationary_vehicle_count = int(round(float(np.mean(stationary_vals)))) if stationary_vals else 0

    # Queue length (integer)
    if avg_speed_kmh < 15 and vehicle_count > 15:
        queue_length_veh = int(round(vehicle_count * 0.50))
    elif avg_speed_kmh < 25 and vehicle_count > 20:
        queue_length_veh = int(round(vehicle_count * 0.30))
    else:
        queue_length_veh = int(round(stationary_vehicle_count))
    queue_length_veh = max(0, queue_length_veh)

    wait_time_min = round(queue_length_veh * 0.4, 2)
    green_seconds = 45.0

    # Density (weighted 3-signal formula)
    # Signal 1: occupancy density from bounding box area
    bbox_areas = [f.get('bbox_area', 0) for f in frame_data]
    avg_bbox_area = float(np.mean(bbox_areas)) if bbox_areas else 0.0
    roi_area = frame_area if frame_area > 0 else 1280 * 720
    occupancy_density = min(avg_bbox_area / max(roi_area, 1), 1.0)

    # Signal 2: count density from vehicle count
    count_density = min(vehicle_count / max(EXPECTED_CAPACITY, 1), 1.0)

    # Signal 3: queue density
    queue_density = queue_length_veh / max(vehicle_count, 1)
    queue_density = min(queue_density, 1.0)

    density_percent = (
        occupancy_density * 0.30 +
        count_density     * 0.50 +
        queue_density     * 0.20
    ) * 100
    density_percent = round(min(100.0, max(0.0, density_percent)), 2)

    # Weather from metadata
    weather      = meta['weather']
    weather_temp = meta['weather_temp_c']

    return {
        'video_name':          result['video_name'],
        'detected_path':       result['detected_path'],
        'date':                meta['date'],
        'time':                meta['time'],
        'hour':                meta['hour'],
        'minute':              meta['minute'],
        'day':                 meta['day'],
        'day_of_week':         meta['day_of_week'],
        'month':               meta['month'],
        'is_holiday':          0,
        'is_weekend':          meta['is_weekend'],
        'hour_sin':            round(np.sin(2*np.pi*meta['hour']/24), 6),
        'hour_cos':            round(np.cos(2*np.pi*meta['hour']/24), 6),
        'vehicle_count':       vehicle_count,
        'vehicle_count_1min':  vehicle_count_1min,
        'volume_veh_per_hour': volume_veh_per_hour,
        'avg_speed_kmh':       avg_speed_kmh,
        'queue_length_veh':    queue_length_veh,
        'wait_time_min':       wait_time_min,
        'green_seconds':       green_seconds,
        'density_percent':     density_percent,
        'weather_temp_c':      weather_temp,
        'weather_condition':   weather,
        'accident_count':      0,
        'roadwork_flag':       0,
        'event_flag':          0,
        # Debug (not in final JSON)
        '_cumulative_count':        cumulative_count,
        '_duration_s':              round(duration_s, 1),
        '_avg_occ':                 round(avg_occ, 3),
        '_slow_ratio':              round(slow_ratio, 3),
        '_stationary_vehicle_count': stationary_vehicle_count,
        '_occupancy_density':       round(occupancy_density, 4),
        '_count_density':           round(count_density, 4),
        '_queue_density':           round(queue_density, 4),
        '_speed_samples':           len(raw_speeds),
        '_weather_source':          meta['weather_source'],
        '_metadata_found':          meta['metadata_found'],
    }

## 16. Process All Selected Videos

In [ ]:
detection_results  = []
video_feature_rows = []
failed_videos      = []

for v in selected_videos:
    print(f"\n{'='*60}")
    print(f"Processing: {v['name']} [{v['period_label']}]")
    print(f"  Meta: date={v['meta']['date']} | time={v['meta']['time']} | "
          f"weather={v['meta']['weather']} {v['meta']['weather_temp_c']}C")

    video_name = os.path.splitext(v['name'])[0]
    result = run_yolo_on_video(v['path'], video_name, v)

    if result is None:
        failed_videos.append(v['name'])
        continue

    result['source_video'] = v['name']
    result['period_label'] = v['period_label']
    detection_results.append(result)

    feat = extract_video_features(result, v)
    if feat is None:
        failed_videos.append(v['name'])
        continue

    feat['source_video'] = v['name']
    feat['period_label'] = v['period_label']
    video_feature_rows.append(feat)

    print(f"  vc_1min={feat['vehicle_count_1min']} | "
          f"speed={feat['avg_speed_kmh']} km/h | "
          f"density={feat['density_percent']}%")

print(f"\nProcessed: {len(detection_results)} | Failed: {len(failed_videos)}")
if failed_videos:
    print(f"Failed: {failed_videos}")
if not detection_results:
    print("[WARNING] No videos processed. Check video paths and Drive mount.")
    print("Suggestion: Reduce MAX_VIDEOS_PER_PERIOD, increase FRAME_STRIDE, "
          "or disable SAVE_DETECTED_VIDEO.")


Processing: pagi_cctv_001_20260518_070935.mp4 [pagi]
  Meta: date=5/18/2026 | time=6:09:33 | weather=Cloudy 25.6C

  [pagi_cctv_001_20260518_070935] DONE
    source_video          : pagi_cctv_001_20260518_070935
    original_fps          : 15.021384610035119
    total_frames_read     : 288
    total_frames_written  : 288
    frame_stride          : 5
    yolo_frames_processed : 58
    unique_vehicles       : 6
    duration_s            : 19.2
    [OK]      frames_written/frames_read = 1.0
    Saved: /content/outputs/detected_videos/detected_pagi_cctv_001_20260518_070935.mp4
  vc_1min=19 | speed=49.02 km/h | density=5.49%

Processing: pagi_cctv_002_20260518_070954.mp4 [pagi]
  Meta: date=5/18/2026 | time=6:15:26 | weather=Cloudy 26.8C

  [pagi_cctv_002_20260518_070954] DONE
    source_video          : pagi_cctv_002_20260518_070954
    original_fps          : 30.04304516494742
    total_frames_read     : 1696
    total_frames_written  : 1696
    frame_stride          : 5
    yolo_frames

## 17. Build Feature DataFrame (Lag & Rolling)

In [ ]:
import pandas as pd

df_feat = pd.DataFrame(video_feature_rows).reset_index(drop=True)

# Weather one-hot
for cond in ['Clear', 'Cloudy', 'Hot', 'Rain']:
    df_feat[f'weather_condition_{cond}'] = (df_feat['weather_condition'] == cond).astype(int)

# Lag/rolling will be computed properly inside build_lstm_sequence from generated history.
# Here we only populate them with the observation-level values as a reference row;
# the actual LSTM input sequence is built per-video in cell 38 with full temporal history.
vol = df_feat['vehicle_count_1min']
df_feat['delta_volume']   = vol.diff().fillna(0)
df_feat['lag_1']          = vol.shift(1).bfill()
df_feat['lag_5']          = vol.shift(5).bfill()
df_feat['lag_15']         = vol.shift(15).bfill()
df_feat['lag_30']         = vol.shift(30).bfill()
df_feat['lag_60']         = vol.shift(60).bfill()
df_feat['lag_speed_15']   = df_feat['avg_speed_kmh'].shift(15).bfill()
df_feat['lag_queue_15']   = df_feat['queue_length_veh'].shift(15).bfill()
df_feat['roll_mean_15']   = vol.rolling(15, min_periods=1).mean()
df_feat['roll_std_15']    = vol.rolling(15, min_periods=1).std().fillna(0)
df_feat['roll_min_15']    = vol.rolling(15, min_periods=1).min()
df_feat['roll_max_15']    = vol.rolling(15, min_periods=1).max()
df_feat['roll_median_15'] = vol.rolling(15, min_periods=1).median()
df_feat['roll_mean_60']   = vol.rolling(60, min_periods=1).mean()
df_feat['roll_std_60']    = vol.rolling(60, min_periods=1).std().fillna(0)
df_feat['roll_min_60']    = vol.rolling(60, min_periods=1).min()
df_feat['roll_max_60']    = vol.rolling(60, min_periods=1).max()

for col in FEATURE_COLUMNS:
    if col not in df_feat.columns:
        df_feat[col] = 0.0

# ── Feature Audit ──────────────────────────────────────────────────────────────
missing_features = [c for c in FEATURE_COLUMNS if c not in df_feat.columns]
extra_features   = [c for c in df_feat.columns
                    if c not in set(FEATURE_COLUMNS)
                    and not c.startswith('_')
                    and c not in ('video_name','source_video','period_label',
                                   'detected_path','date','time','weather_condition')]

# Validate feature order
inf_order   = [c for c in df_feat.columns if c in set(FEATURE_COLUMNS)]
order_match = inf_order == FEATURE_COLUMNS

print("=== FEATURE AUDIT ===")
print(f"training_feature_count   : {len(FEATURE_COLUMNS)}")
print(f"inference_feature_count  : {len(inf_order)}")
print(f"missing_features         : {missing_features if missing_features else 'None'}")
print(f"extra_features (ignored) : {len(extra_features)}")
print(f"feature_order_match      : {order_match}")

if missing_features:
    raise RuntimeError(
        f"[STOP] {len(missing_features)} features missing from inference DataFrame: "
        f"{missing_features}. Cannot proceed."
    )

print("\n[OK] All training features present.")
print(f"Feature DataFrame: {df_feat.shape}")
print(df_feat[['video_name','vehicle_count_1min','avg_speed_kmh',
               'density_percent','date','time','weather_condition']].to_string())

=== FEATURE AUDIT ===
training_feature_count   : 41
inference_feature_count  : 41
missing_features         : None
extra_features (ignored) : 1
feature_order_match      : False

[OK] All training features present.
Feature DataFrame: (21, 60)
                        video_name  vehicle_count_1min  avg_speed_kmh  density_percent       date         time weather_condition
0    pagi_cctv_001_20260518_070935                  19          49.02             5.49  5/18/2026      6:09:33            Cloudy
1    pagi_cctv_002_20260518_070954                  35          45.07            28.53  5/18/2026      6:15:26            Cloudy
2    pagi_cctv_004_20260518_071356                  22          42.65            16.34  5/18/2026      6:18:01            Cloudy
3    pagi_cctv_006_20260518_071450                  36          20.22            37.50  5/18/2026      6:20:24            Cloudy
4    pagi_cctv_009_20260518_071625                  64          44.89            13.96  5/18/2026      6:20:49    

## 18. Save Traffic Features CSV

In [ ]:
csv_path = os.path.join(OUTPUT_DIR, 'traffic_features_inference.csv')
df_feat.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

Saved: /content/outputs/traffic_features_inference.csv


## 19. LSTM Sequence Builder & Inverse Transform

In [ ]:
from datetime import datetime, timedelta
import math

def video_name_to_seed(video_name):
    return abs(hash(video_name)) % (2**31)


def build_training_aligned_sequence(obs, seq_len, rng):
    vc   = float(obs.get('vehicle_count_1min', 30.0))
    vol  = vc * 60.0
    spd  = float(obs.get('avg_speed_kmh', 50.0))
    q    = float(obs.get('queue_length_veh', 0.0))
    den  = float(obs.get('density_percent', 20.0))

    # Reconstruct base datetime (step i=seq_len-1 is "now")
    try:
        obs_dt = datetime(
            int(obs.get('year', 2024)), int(obs.get('month', 5)),
            int(obs.get('day', 1)), int(obs.get('hour', 8)), int(obs.get('minute', 0))
        )
    except Exception:
        obs_dt = datetime(2024, 5, 1, int(obs.get('hour', 8)), int(obs.get('minute', 0)))

    # Step 1: Generate 60-step volume_veh_per_hour history
    # BUG-A FIX: when vol=0, keep history at 0 (no noise inflation)
    noise_pct = 0.04
    vols = []
    if vol <= 0:
        vols = [0.0] * seq_len
    else:
        for i in range(seq_len):
            ramp  = 0.70 + 0.30 * (i / max(seq_len - 1, 1))
            base  = vol * ramp
            noise = rng.normal(0, max(abs(base) * noise_pct, 5.0))
            vols.append(max(0.0, base + noise))
        vols[-1] = vol    # anchor last step to actual observation

    vol_s = np.array(vols)

    # BUG-B FIX: safe denominator prevents explosion when vol=0 but q or den > 0
    vol_denom = max(vol, 1.0)

    SPEED_BASE = max(spd, 10.0)
    VOL_BASE   = max(vol, 60.0)
    spds = [max(5.0, SPEED_BASE * (1.0 - 0.4 * (vol_s[i] - vol_s[-1]) / VOL_BASE))
            for i in range(seq_len)]
    spds[-1] = spd

    qs   = [q * (vol_s[i] / vol_denom) for i in range(seq_len)]
    qs[-1] = q
    dens = [min(100.0, den * (vol_s[i] / vol_denom)) for i in range(seq_len)]
    dens[-1] = den

    # Step 2: Build per-timestep feature rows
    rows = []
    for i in range(seq_len):
        vc_i  = vol_s[i] / 60.0
        vol_i = vol_s[i]
        spd_i = spds[i]
        q_i   = qs[i]
        den_i = dens[i]

        wait_i  = q_i * 0.5
        green_i = float(obs.get('green_seconds', 25.6))

        dv = vol_s[i] - vol_s[i-1] if i > 0 else 0.0

        def lv(lag): return float(vol_s[max(i - lag, 0)])
        lag1  = lv(1);  lag5  = lv(5)
        lag15 = lv(15); lag30 = lv(30); lag60 = lv(60)

        lag_spd15 = float(spds[max(i-15, 0)])
        lag_q15   = float(qs[max(i-15, 0)])

        w15 = vol_s[max(0, i-14):i+1]
        w60 = vol_s[max(0, i-59):i+1]

        # BUG-C FIX: reconstruct per-step timestamp
        # step i=seq_len-1 is "now"; earlier steps go back in time
        minutes_ago = (seq_len - 1) - i
        step_dt     = obs_dt - timedelta(minutes=minutes_ago)
        h_i         = step_dt.hour
        m_i         = step_dt.minute
        sin_i       = math.sin(2 * math.pi * h_i / 24)
        cos_i       = math.cos(2 * math.pi * h_i / 24)

        row = {
            'vehicle_count_1min':   vc_i,
            'volume_veh_per_hour':  vol_i,
            'avg_speed_kmh':        spd_i,
            'queue_length_veh':     q_i,
            'wait_time_min':        wait_i,
            'green_seconds':        green_i,
            'density_percent':      den_i,
            'weather_temp_c':       float(obs.get('weather_temp_c', 30.0)),
            'accident_count':       float(obs.get('accident_count', 0)),
            'roadwork_flag':        float(obs.get('roadwork_flag', 0)),
            'event_flag':           float(obs.get('event_flag', 0)),
            'hour':                 float(h_i),
            'minute':               float(m_i),
            'day':                  float(obs.get('day', 1)),
            'day_of_week':          float(obs.get('day_of_week', 1)),
            'month':                float(obs.get('month', 6)),
            'is_holiday':           float(obs.get('is_holiday', 0)),
            'is_weekend':           float(obs.get('is_weekend', 0)),
            'hour_sin':             sin_i,
            'hour_cos':             cos_i,
            'delta_volume':         dv,
            'lag_1':               lag1,
            'lag_5':               lag5,
            'lag_15':              lag15,
            'lag_30':              lag30,
            'lag_60':              lag60,
            'lag_speed_15':        lag_spd15,
            'lag_queue_15':        lag_q15,
            'roll_mean_15':        float(w15.mean()),
            'roll_std_15':         float(w15.std()) if len(w15) > 1 else 0.0,
            'roll_min_15':         float(w15.min()),
            'roll_max_15':         float(w15.max()),
            'roll_median_15':      float(np.median(w15)),
            'roll_mean_60':        float(w60.mean()),
            'roll_std_60':         float(w60.std()) if len(w60) > 1 else 0.0,
            'roll_min_60':         float(w60.min()),
            'roll_max_60':         float(w60.max()),
            'weather_condition_Clear':  float(obs.get('weather_condition_Clear', 1)),
            'weather_condition_Cloudy': float(obs.get('weather_condition_Cloudy', 0)),
            'weather_condition_Hot':    float(obs.get('weather_condition_Hot', 0)),
            'weather_condition_Rain':   float(obs.get('weather_condition_Rain', 0)),
        }
        rows.append(row)

    # Step 3: Assemble array in exact FEATURE_COLUMNS order
    arr = np.array([[r[col] for col in FEATURE_COLUMNS] for r in rows], dtype=np.float64)
    return arr    # shape: (seq_len, n_features)

def predict_from_obs(obs, video_name):
    rng  = np.random.default_rng(video_name_to_seed(video_name))
    seq  = build_training_aligned_sequence(obs, SEQ_LEN, rng)  # (60, 41)

    # Diagnostic output
    expected_feat_count = len(FEATURE_COLUMNS)
    print(f"    raw_feature_shape            : {seq.shape}  "
          f"expected ({SEQ_LEN}, {expected_feat_count})")

    vol_col  = FEATURE_COLUMNS.index('volume_veh_per_hour')
    lag1_col = FEATURE_COLUMNS.index('lag_1')
    rm60_col = FEATURE_COLUMNS.index('roll_mean_60')
    print(f"    vol_range_in_seq (veh/hr)    : {seq[:, vol_col].min():.1f} - {seq[:, vol_col].max():.1f}")
    print(f"    lag_1 at last step           : {seq[-1, lag1_col]:.1f}")
    print(f"    roll_mean_60 at last step    : {seq[-1, rm60_col]:.1f}")

    if seq.shape != (SEQ_LEN, expected_feat_count):
        raise ValueError(
            f"[ERROR] raw sequence shape {seq.shape} != "
            f"({SEQ_LEN}, {expected_feat_count}). Check FEATURE_COLUMNS."
        )

    if not np.isfinite(seq).all():
        n_bad = (~np.isfinite(seq)).sum()
        bad_cols = [FEATURE_COLUMNS[j] for j in range(seq.shape[1])
                    if not np.isfinite(seq[:, j]).all()]
        print(f"    [WARNING] {n_bad} non-finite values in columns: {bad_cols}")
        seq = np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0)

    scaled = feat_scaler.transform(seq)

    if scaled.shape[1] != expected_feat_count:
        raise ValueError(
            f"[ERROR] scaled shape {scaled.shape} — "
            f"scaler output dim {scaled.shape[1]} != {expected_feat_count}."
        )
    print(f"    scaled_feature_shape         : {scaled.shape}")

    X = scaled.reshape(1, SEQ_LEN, NUM_FEATURES).astype(np.float32)
    print(f"    sequence_shape_after_reshape : {X.shape}  "
          f"expected (1, {SEQ_LEN}, {NUM_FEATURES})")

    pred_scaled = lstm_model.predict(X, verbose=0)
    print(f"    raw_model_output             : {pred_scaled.flatten()[0]:.6f}")

    ts = target_scalers
    if isinstance(ts, dict):
        if '15m' not in ts:
            raise ValueError(f"[ERROR] target_scalers has no '15m' key. Keys: {list(ts.keys())}")
        scaler_15m = ts['15m']
    else:
        scaler_15m = ts
    print(f"    target_scaler_type           : {type(scaler_15m).__name__}")

    inv = float(scaler_15m.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()[0])
    print(f"    inverse_transformed_output   : {inv:.2f} veh/hour")

    return max(0.0, round(inv, 2))


# Sensitivity Test
print("=== LSTM SENSITIVITY TEST (Fixed v2) ===")
print("Expected: predictions vary proportionally with vehicle_count_1min")
print()

_base_hour = 8
_base_obs  = {
    'avg_speed_kmh':    50.0,
    'queue_length_veh': 2.0,
    'density_percent':  20.0,
    'weather_temp_c':   30.0,
    'hour':             _base_hour,
    'minute':           0,
    'day':              18,
    'day_of_week':      0,
    'month':            5,
    'is_holiday':       0,
    'is_weekend':       0,
    'hour_sin':         float(np.sin(2*np.pi*_base_hour/24)),
    'hour_cos':         float(np.cos(2*np.pi*_base_hour/24)),
    'weather_condition_Clear': 1, 'weather_condition_Cloudy': 0,
    'weather_condition_Hot': 0,   'weather_condition_Rain': 0,
    'accident_count': 0, 'roadwork_flag': 0, 'event_flag': 0,
    'green_seconds': 25.6, 'wait_time_min': 1.0,
}

_scenarios = [
    ('A',   0, "empty road"),
    ('B',  20, "light traffic"),
    ('C',  40, "moderate traffic"),
    ('D',  80, "heavy traffic"),
    ('E', 120, "very heavy traffic"),
]

_preds  = []
_vc_vals = []
print(f"  {'Scenario':<28} {'vc_1min':>8}  {'vol/hr':>9}  {'pred_15m':>12}")
print(f"  {'-'*28} {'-'*8}  {'-'*9}  {'-'*12}")
for label, vc_val, desc in _scenarios:
    obs = dict(_base_obs)
    obs['vehicle_count_1min']  = float(vc_val)
    obs['volume_veh_per_hour'] = float(vc_val * 60)
    obs['queue_length_veh']    = max(0.0, vc_val * 0.15)
    obs['density_percent']     = min(100.0, vc_val / 1.7)
    obs['avg_speed_kmh']       = max(10.0, 65.0 - vc_val * 0.35)
    obs['wait_time_min']       = max(0.0, vc_val * 0.07)
    val = predict_from_obs(obs, f"sensitivity_{label}")
    _preds.append(val)
    _vc_vals.append(vc_val)
    print(f"  {label}: {desc:<24} {vc_val:>8.0f}  {vc_val*60:>9.0f}  {val:>12.1f}")

_spread = max(_preds) - min(_preds)
_mean   = sum(_preds) / len(_preds)
_rel    = _spread / max(abs(_mean), 1.0)
print()
if _rel < 0.05:
    print("[WARNING] LSTM predictions are nearly constant. "
          "Inference pipeline still does not match training behavior.")
else:
    print(f"[OK] Sensitivity spread = {round(_spread, 1)} veh/hour "
          f"({round(_rel*100, 1)}% relative). Model responds to traffic load.")
print()

print("=== DEBUG REPORT ===")
print(f"  Feature validation  : {'OK - all ' + str(len(FEATURE_COLUMNS)) + ' features present'}")
print(f"  Scaler type         : {type(feat_scaler).__name__} | "
      f"n_features_in={getattr(feat_scaler, 'n_features_in_', 'N/A')}")
print(f"  Target scaler       : {type(list(target_scalers.values())[0]).__name__} "
      f"(key='15m', center={list(target_scalers.values())[0].center_})")
print(f"  Sequence shape      : ({SEQ_LEN}, {NUM_FEATURES}) -> (1, {SEQ_LEN}, {NUM_FEATURES})")
print(f"  Sensitivity test    : vc range {min(_vc_vals)}-{max(_vc_vals)} "
      f"-> pred range {round(min(_preds),1)}-{round(max(_preds),1)} veh/hr")
print(f"  Fixes applied       : BUG-A (noise floor), BUG-B (div/zero), BUG-C (frozen time)")
print("LSTM helpers ready.")

=== LSTM SENSITIVITY TEST (Fixed v2) ===
Expected: predictions vary proportionally with vehicle_count_1min

  Scenario                      vc_1min     vol/hr      pred_15m
  ---------------------------- --------  ---------  ------------
    raw_feature_shape            : (60, 41)  expected (60, 41)
    vol_range_in_seq (veh/hr)    : 0.0 - 0.0
    lag_1 at last step           : 0.0
    roll_mean_60 at last step    : 0.0
    scaled_feature_shape         : (60, 41)
    sequence_shape_after_reshape : (1, 60, 41)  expected (1, 60, 41)
    raw_model_output             : 0.156860
    target_scaler_type           : RobustScaler
    inverse_transformed_output   : 1905.58 veh/hour
  A: empty road                      0          0        1905.6
    raw_feature_shape            : (60, 41)  expected (60, 41)
    vol_range_in_seq (veh/hr)    : 834.0 - 1292.7
    lag_1 at last step           : 1292.7
    roll_mean_60 at last step    : 1021.6
    scaled_feature_shape         : (60, 41)
    sequence_s

## 20. Per-Video LSTM Prediction

In [ ]:
print("=== PER-VIDEO LSTM PREDICTION ===")

for result in detection_results:
    vname    = result['video_name']
    feat_row = df_feat[df_feat['video_name'] == vname]
    if feat_row.empty:
        result['predicted_volume_15m'] = 0.0
        print(f"  [SKIP] {vname} — no feature row")
        continue

    r = feat_row.iloc[0]

    # Build obs dict from actual video features
    # Includes weather one-hot and all context fields
    obs = {col: float(r[col]) if col in r.index else 0.0
           for col in FEATURE_COLUMNS}

    # Ensure weather one-hot matches weather_condition string
    weather_str = str(r.get('weather_condition', 'Clear'))
    for cond in ['Clear', 'Cloudy', 'Hot', 'Rain']:
        obs[f'weather_condition_{cond}'] = 1.0 if weather_str == cond else 0.0

    print(f"  {vname}")
    print(f"    vehicle_count_1min  = {obs['vehicle_count_1min']:.2f}")
    print(f"    volume_veh_per_hour = {obs['volume_veh_per_hour']:.2f}")
    print(f"    avg_speed_kmh       = {obs['avg_speed_kmh']:.2f}")
    print(f"    density_percent     = {obs['density_percent']:.2f}")

    try:
        val = predict_from_obs(obs, vname)
    except Exception as e:
        print(f"    [ERROR] {e}")
        val = 0.0

    result['predicted_volume_15m'] = val
    print(f"    -> predicted_volume_15m = {val} veh/hour")

=== PER-VIDEO LSTM PREDICTION ===
  pagi_cctv_001_20260518_070935
    vehicle_count_1min  = 19.00
    volume_veh_per_hour = 1140.00
    avg_speed_kmh       = 49.02
    density_percent     = 5.49
    raw_feature_shape            : (60, 41)  expected (60, 41)
    vol_range_in_seq (veh/hr)    : 753.3 - 1161.6
    lag_1 at last step           : 1075.7
    roll_mean_60 at last step    : 973.7
    scaled_feature_shape         : (60, 41)
    sequence_shape_after_reshape : (1, 60, 41)  expected (1, 60, 41)
    raw_model_output             : 0.264099
    target_scaler_type           : RobustScaler
    inverse_transformed_output   : 2027.30 veh/hour
    -> predicted_volume_15m = 2027.3 veh/hour
  pagi_cctv_002_20260518_070954
    vehicle_count_1min  = 35.00
    volume_veh_per_hour = 2100.00
    avg_speed_kmh       = 45.07
    density_percent     = 28.53
    raw_feature_shape            : (60, 41)  expected (60, 41)
    vol_range_in_seq (veh/hr)    : 1470.6 - 2193.6
    lag_1 at last step        

## 21. Build Per-Video Prediction Output JSON

In [ ]:
def get_congestion_level(density_percent, avg_speed_kmh, vehicle_count, queue_length_veh):
    # Normalization
    density = density_percent / 100
    speed = min(avg_speed_kmh, 80)        # cap speed untuk stabilitas model
    speed_norm = speed / 80

    # Non-linear pressure signals
    speed_pressure = (1 - speed_norm) ** 2.0   # semakin lambat → eksponensial naik
    queue_pressure = min(queue_length_veh / 8, 1)  # bottleneck indicator
    vehicle_pressure = min(vehicle_count / 70, 1)   # kapasitas jalan

    # Base pressure model
    pressure = (
        0.30 * density +
        0.40 * speed_pressure +
        0.20 * queue_pressure +
        0.10 * vehicle_pressure
    )

    # Hard safety rules
    if speed < 1 and density < 0.02:
        return "Low"   # sensor noise / idle condition

    if speed < 5:
        if queue_length_veh >= 2 or density > 0.2:
            return "Severe"  # breakdown traffic
        return "High"       # near-stall condition

    # Synergy effect (real traffic behavior)
    if speed < 20 and density > 0.4:
        pressure += 0.18   # congestion amplification

    if speed < 15 and queue_length_veh > 0:
        pressure += 0.12   # bottleneck effect

    if density > 0.55 and speed < 40:
        pressure += 0.10   # unstable flow region

    # Anti false positive fix
    if speed > 60 and density > 0.5:
        pressure -= 0.12   # fast but dense correction

    if speed < 10:
        pressure += 0.20   # forced congestion detection

    # Contradiction/ Intelligence layer
    if density > 0.45 and speed > 60:
        pressure += 0.20   # unstable fast-dense flow

    if density > 0.5 and queue_length_veh == 0 and speed < 35:
        pressure += 0.10   # hidden congestion detection

    # Final classification
    if pressure >= 0.58:
        return "Severe"
    elif pressure >= 0.44:
        return "High"
    elif pressure >= 0.28:
        return "Medium"
    else:
        return "Low"

# FLOW INTERPRETATION ENGINE
def interpret_flow(density, speed, queue):
    # dense but fast flow (high throughput highway state)
    if density > 0.6 and speed > 45:
        return "dense_fast_flow"

    # congested traffic state
    elif density > 0.55 and speed < 25:
        return "congested_flow"

    # traffic breakdown (near stop)
    elif speed < 8 and queue > 0:
        return "breakdown_flow"

    # unstable transition flow
    elif 0.35 < density < 0.6 and 25 <= speed <= 55:
        return "unstable_flow"

    # free flow condition
    elif density < 0.25 and speed > 50:
        return "free_flow"

    # default normal state
    return "normal_flow"

def build_ai_insight(vc, spd, density, weather, period, queue_len, congestion_level, accident, roadwork, event):
    parts = []

    density_ratio = density / 100
    flow_type = interpret_flow(density_ratio, spd, queue_len)

    if congestion_level == "Severe":
        parts.append(
            f"Severe congestion detected with {queue_len} queued vehicles. "
            f"Speed dropped to {round(spd,1)} km/h indicating traffic breakdown."
        )

    elif congestion_level == "High":
        if flow_type == "critical_queue_flow":
            parts.append("High congestion caused by queue spillback and bottleneck failure.")
        else:
            parts.append(
                f"High traffic density with {queue_len} queued vehicles and speed {round(spd,1)} km/h."
            )

    elif congestion_level == "Medium":
        if flow_type in ["dense_flow", "unstable_flow"]:
            parts.append("Traffic is dense but still maintaining partial flow stability.")
        else:
            parts.append(
                f"Moderate congestion with {vc} vehicles and speed {round(spd,1)} km/h."
            )

    else:
        parts.append(
            f"Stable traffic with low density ({round(density,1)}%) and speed {round(spd,1)} km/h."
        )

    # context layer
    if period in ("Morning Rush", "Evening Rush"):
        parts.append(f"Peak-hour ({period}) increases traffic demand.")

    if weather == "Rain":
        parts.append("Rain may reduce road friction and visibility.")

    if accident > 0:
        parts.append(f"{accident} accident(s) impacting flow.")

    if roadwork:
        parts.append("Roadwork reduces lane capacity.")

    if event:
        parts.append("Nearby event increases traffic volume.")

    return " ".join(parts)

def build_recommendation(congestion, queue, density, speed):
    actions = []

    if congestion == "Severe":
        actions.append("Activate emergency traffic control and rerouting.")
        if speed < 5:
            actions.append("Deploy rapid incident response team.")
        if queue > 3:
            actions.append("Clear queue buildup immediately.")

    elif congestion == "High":
        actions.append("Increase signal optimization and corridor monitoring.")
        if queue > 5:
            actions.append("Focus on bottleneck queue reduction.")
        if density > 60:
            actions.append("Adjust adaptive signal timing.")

    elif congestion == "Medium":
        actions.append("Monitor closely and prepare early intervention.")
        if speed < 30:
            actions.append("Consider pre-emptive signal adjustment.")

    else:
        actions.append("Maintain normal operations and routine monitoring.")

    return " ".join(actions)

# MAIN PIPELINE
prediction_records = []
failed_videos = []
video_feature_rows = []  # supaya validation tidak error

for idx, result in enumerate(detection_results):
    vname = result['video_name']
    feat_row = df_feat[df_feat['video_name'] == vname]

    if feat_row.empty:
        failed_videos.append(vname)
        continue

    r = feat_row.iloc[0]

    vc = int(r['vehicle_count'])

    vc_1min = int(r['vehicle_count_1min']) if 'vehicle_count_1min' in r else vc
    vol = int(r['volume_veh_per_hour']) if 'volume_veh_per_hour' in r else vc * 6

    spd = float(r['avg_speed_kmh'])
    density = float(r['density_percent'])
    queue = int(r['queue_length_veh'])
    weather = str(r['weather_condition'])
    hour = int(r['hour'])
    period = get_period_label(hour)

    cong = get_congestion_level(density, spd, vc, queue)

    insight = build_ai_insight(
        vc, spd, density, weather, period, queue,
        cong,
        int(r.get('accident_count', 0)),
        int(r.get('roadwork_flag', 0)),
        int(r.get('event_flag', 0))
    )

    recom = build_recommendation(cong, queue, density, spd)

    rec = {
      "camera_id": f"CAM_{idx+1:03d}",
      "source_video": result['source_video'],

      # TIME INFO (WAJIB)
      "date": str(r['date']),
      "time": str(r['time']),
      "period": period,

      # LOCATION INFO
      "latitude": meta.get('lat'),
      "longitude": meta.get('lon'),

      # WEATHER INFO
      "weather": weather,
      "weather_temp_c": float(r['weather_temp_c']),

      # TRAFFIC CORE
      "vehicle_count": vc,
      "vehicle_count_1min": int(r['vehicle_count_1min']),
      "volume_veh_per_hour": int(r['volume_veh_per_hour']),
      "avg_speed_kmh": round(spd, 2),
      "density_percent": round(density, 2),
      "queue_length_veh": queue,

      # MODEL OUTPUT
      "congestion_level": cong,
      "predicted_volume_15m": result['predicted_volume_15m'],

      # METADATA
      "model_mape_15m": MODEL_MAPE,
      "ai_insight": insight,
      "recommendation": recom
}

    prediction_records.append(rec)
    video_feature_rows.append(rec)


print(f"Records built: {len(prediction_records)}")

# OUTPUT
for rec in prediction_records:
    print(
        rec['camera_id'],
        rec['source_video'],
        "| vc =", rec['vehicle_count'],
        "| spd =", rec['avg_speed_kmh'],
        "| que =", rec['queue_length_veh'],
        "| cong =", rec['congestion_level'],
        "| insight =", rec['ai_insight'],
        "| recom =", rec['recommendation']
    )

Records built: 21
CAM_001 pagi_cctv_001_20260518_070935.mp4 | vc = 6 | spd = 49.02 | que = 0 | cong = Low | insight = Stable traffic with low density (5.5%) and speed 49.0 km/h. Peak-hour (Morning Rush) increases traffic demand. | recom = Maintain normal operations and routine monitoring.
CAM_002 pagi_cctv_002_20260518_070954.mp4 | vc = 33 | spd = 45.07 | que = 0 | cong = Low | insight = Stable traffic with low density (28.5%) and speed 45.1 km/h. Peak-hour (Morning Rush) increases traffic demand. | recom = Maintain normal operations and routine monitoring.
CAM_003 pagi_cctv_004_20260518_071356.mp4 | vc = 19 | spd = 42.65 | que = 0 | cong = Low | insight = Stable traffic with low density (16.3%) and speed 42.6 km/h. Peak-hour (Morning Rush) increases traffic demand. | recom = Maintain normal operations and routine monitoring.
CAM_004 pagi_cctv_006_20260518_071450.mp4 | vc = 37 | spd = 20.22 | que = 11 | cong = Severe | insight = Severe congestion detected with 11 queued vehicles. Speed

## 22. Save Prediction Output JSON

In [ ]:
json_path = os.path.join(OUTPUT_DIR, 'prediction_output.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(prediction_records, f, ensure_ascii=False, indent=2)
print(f"Saved: {json_path}")

Saved: /content/outputs/prediction_output.json


## 23. Save Prediction Output CSV

In [ ]:
pred_df  = pd.DataFrame(prediction_records)
pred_csv = os.path.join(OUTPUT_DIR, 'prediction_output.csv')
pred_df.to_csv(pred_csv, index=False)
print(f"Saved: {pred_csv}")

Saved: /content/outputs/prediction_output.csv


## 24. Validation Summary

In [ ]:
print("\n=== VALIDATION SUMMARY ===")
print(f"Videos processed          : {len(prediction_records)}")
print(f"Videos failed             : {len(failed_videos)}")
if failed_videos:
    print(f"  {failed_videos}")

if len(prediction_records) == 0:
    print("[ERROR] No prediction records. Check video paths and Drive mount.")
else:
    if detection_results:
        print("\nPer-video frame audit:")
        for r in detection_results:
            fr = r.get('total_frames', 0)
            fw = r.get('frames_written', 0)
            fy = r.get('yolo_frames', 0)
            ratio = fw / max(fr, 1)
            ok    = '[OK]     ' if ratio >= 0.80 else '[WARNING]'
            print(f"  {ok} {r['video_name']}")
            print(f"           frames_read={fr} | frames_written={fw} | "
                  f"yolo_processed={fy} | ratio={round(ratio,3)}")
            if ratio < 0.80:
                print(f"           -> Video output may be sped up.")

    print("\n=== PER-VIDEO DETECTION REVIEW ===")
    col_hdr = f"{'source_video':<35} {'vc':>5} {'spd':>7} {'queue':>6} {'density':>8} {'cong':<8} ai_insight[:80]"
    print(col_hdr)
    print("-" * 130)
    for rec in prediction_records:
        src   = rec['source_video'][:33]
        vc    = rec['vehicle_count']
        spd   = rec['avg_speed_kmh']
        queue = rec['queue_length_veh']
        den   = rec['density_percent']
        cong  = rec['congestion_level']
        ins   = rec['ai_insight'][:80]
        print(f"{src:<35} {vc:<5} {spd:>7.1f} {queue:>6} {den:>8.2f} {cong:<8} {ins}")

    print("\n=== TYPE CHECKS ===")
    all_ok = True
    for rec in prediction_records:
        src = rec['source_video']
        vc   = rec['vehicle_count']
        vc1  = rec.get('vehicle_count_1min', vc)
        vol  = rec.get('volume_veh_per_hour', vc)
        q    = rec['queue_length_veh']

        if not isinstance(vc, int):
            print(f"[FAIL] {src}: vehicle_count={vc}"); all_ok=False
        if not isinstance(vc1, int):
            print(f"[FAIL] {src}: vehicle_count_1min={vc1}"); all_ok=False
        if not isinstance(vol, int):
            print(f"[FAIL] {src}: volume_veh_per_hour={vol}"); all_ok=False
        if not isinstance(q, int):
            print(f"[FAIL] {src}: queue_length_veh={q}"); all_ok=False

    if all_ok:
        print("[OK] All vehicle_count, vehicle_count_1min, volume_veh_per_hour, queue_length_veh are int.")

    print("\n=== CONGESTION LOGIC CHECKS ===")
    logic_ok = True
    for rec in prediction_records:
        src  = rec['source_video']
        vc   = rec['vehicle_count']
        spd  = rec['avg_speed_kmh']
        den  = rec['density_percent']
        q    = rec['queue_length_veh']
        cong = rec['congestion_level']

        if spd < 15 and vc > 15 and cong == "Low":
            print(f"[FAIL] {src}: spd={spd}, vc={vc} but congestion_level=Low"); logic_ok=False

        if vc > 40 and den < 20:
            print(f"[WARN] {src}: vc={vc} but density_percent={den}")

        if q >= 20 and spd < 20 and cong not in ("Severe",):
            print(f"[FAIL] {src}: queue={q}, spd={spd} should be Severe, got {cong}"); logic_ok=False

    if logic_ok:
        print("[OK] All congestion logic checks passed.")

    vc_vals   = [r['vehicle_count'] for r in prediction_records]
    spd_vals  = [r['avg_speed_kmh'] for r in prediction_records]
    den_vals  = [r['density_percent'] for r in prediction_records]
    pred_vals = [r.get('predicted_volume_15m', 0) for r in prediction_records]
    weathers  = [r.get('weather', "Unknown") for r in prediction_records]
    cong_dist = [r['congestion_level'] for r in prediction_records]

    print(f"\nvehicle_count    : min={min(vc_vals)}  max={max(vc_vals)}")
    print(f"density_percent  : min={min(den_vals):.2f}  max={max(den_vals):.2f}")
    print(f"avg_speed_kmh    : min={min(spd_vals):.2f}  max={max(spd_vals):.2f}")
    print(f"predicted_vol_15m: min={min(pred_vals):.2f}  max={max(pred_vals):.2f}")

    from collections import Counter
    print(f"\nCongestion distribution:")
    for level, cnt in Counter(cong_dist).most_common():
        print(f"  {level}: {cnt}")

    print(f"\nWeather distribution (from CSV):")
    for w, cnt in Counter(weathers).items():
        print(f"  {w}: {cnt}")

    no_meta = sum(1 for r in video_feature_rows if not r.get('_metadata_found', True))
    print(f"\nVideos without CSV metadata match: {no_meta}")

    def check_variance(label, vals, tol=0.02):
        if len(vals) < 2:
            return
        spread = max(vals) - min(vals)
        rel = spread / max(abs(float(np.mean(vals))), 1e-9)

        if rel < tol:
            print(f"[WARNING] {label}: values nearly identical (spread={round(spread,4)})")
        else:
            print(f"[OK] {label}: spread={round(spread,2)}")

    print()
    check_variance("vehicle_count", vc_vals)
    check_variance("avg_speed_kmh", spd_vals)
    check_variance("density_percent", den_vals)


=== VALIDATION SUMMARY ===
Videos processed          : 21
Videos failed             : 0

Per-video frame audit:
  [OK]      pagi_cctv_001_20260518_070935
           frames_read=288 | frames_written=288 | yolo_processed=58 | ratio=1.0
  [OK]      pagi_cctv_002_20260518_070954
           frames_read=1696 | frames_written=1696 | yolo_processed=300 | ratio=1.0
  [OK]      pagi_cctv_004_20260518_071356
           frames_read=1302 | frames_written=1302 | yolo_processed=261 | ratio=1.0
  [OK]      pagi_cctv_006_20260518_071450
           frames_read=1860 | frames_written=1860 | yolo_processed=300 | ratio=1.0
  [OK]      pagi_cctv_009_20260518_071625
           frames_read=420 | frames_written=420 | yolo_processed=84 | ratio=1.0
  [OK]      pagi_cctv_014_20260518_072255
           frames_read=496 | frames_written=496 | yolo_processed=100 | ratio=1.0
  [OK]      pagi_cctv_015_20260518_072703
           frames_read=3944 | frames_written=3944 | yolo_processed=300 | ratio=1.0
  [OK]      siang_cc

## 25. LSTM Prediction Validation

In [ ]:
print("=== LSTM PREDICTION VALIDATION ===")
videos = []

for rec in prediction_records:
    pred = rec["predicted_volume_15m"]

    lower = pred * (1 - MODEL_MAPE / 100)
    upper = pred * (1 + MODEL_MAPE / 100)

    videos.append({
        "camera_id": rec.get("camera_id"),
        "source_video": rec.get("source_video"),

        "date": rec.get("date", "unknown"),
        "time": rec.get("time", "unknown"),
        "period": rec.get("period", "unknown"),

        "latitude": rec.get("latitude", None),
        "longitude": rec.get("longitude", None),

        "weather": rec.get("weather", "unknown"),
        "weather_temp_c": rec.get("weather_temp_c", None),

        "vehicle_count": rec.get("vehicle_count"),
        "avg_speed_kmh": rec.get("avg_speed_kmh"),
        "density_percent": rec.get("density_percent"),

        "congestion_level": rec.get("congestion_level"),

        "predicted_volume_15m": float(pred),

        "confidence_range": {
            "lower_bound": float(lower),
            "upper_bound": float(upper)
        }
    })

    print(
        f"{rec.get('source_video','')[:35]:<35} "
        f"pred={pred:.0f} veh/hr | "
        f"confidence_range=({lower:.0f}-{upper:.0f})"
    )

print(f"\nModel MAPE : {MODEL_MAPE:.2f}%")
print("Confidence range based on historical error (MAPE).")

# =====================================
print("\n=== PREDICTION DISTRIBUTION ANALYSIS ===")

preds = np.array([v["predicted_volume_15m"] for v in videos])

pred_min = float(np.min(preds))
pred_max = float(np.max(preds))
pred_mean = float(np.mean(preds))
pred_std = float(np.std(preds))

pred_spread = pred_max - pred_min
relative_spread = pred_spread / max(pred_mean, 1)

print(f"Min Prediction   : {pred_min:.2f}")
print(f"Max Prediction   : {pred_max:.2f}")
print(f"Mean Prediction  : {pred_mean:.2f}")
print(f"Std Deviation    : {pred_std:.2f}")
print(f"Prediction Spread: {pred_spread:.2f}")

# =====================================
print("\n=== PREDICTION VARIABILITY CHECK ===")
print(f"Relative Spread : {relative_spread:.2%}")

if relative_spread < 0.10:
    print("[WARNING] Predictions are highly concentrated.")
elif relative_spread < 0.20:
    print("[INFO] Predictions show moderate variability.")
else:
    print("[OK] Predictions show strong variability.")

# =====================================
print("\n=== PREDICTION VS VEHICLE COUNT ===")

data_sorted = sorted(
    [(v["vehicle_count"], v["predicted_volume_15m"]) for v in videos],
    key=lambda x: x[0]
)

vehicle_counts = np.array([x[0] for x in data_sorted])
pred_values = np.array([x[1] for x in data_sorted])

print(f"{'vehicle_count':>15} {'prediction':>15}")
print("-" * 35)

for vc, pv in data_sorted:
    print(f"{vc:>15} {pv:>15.2f}")

# =====================================
# CORRELATION (RETURN VALUE, NOT GLOBAL SIDE EFFECT)
corr_value = None

if len(np.unique(vehicle_counts)) > 1:
    corr_value = float(np.corrcoef(vehicle_counts, pred_values)[0, 1])
    print(f"\nCorrelation: {corr_value:.3f}")

    if abs(corr_value) < 0.20:
        print("[WARNING] Weak relationship detected.")
    elif abs(corr_value) < 0.50:
        print("[INFO] Moderate relationship detected.")
    else:
        print("[OK] Strong relationship detected.")
else:
    print("\n[INFO] Correlation undefined (no variance in vehicle_count)")

=== LSTM PREDICTION VALIDATION ===
pagi_cctv_001_20260518_070935.mp4   pred=2027 veh/hr | confidence_range=(1610-2445)
pagi_cctv_002_20260518_070954.mp4   pred=2025 veh/hr | confidence_range=(1608-2443)
pagi_cctv_004_20260518_071356.mp4   pred=2039 veh/hr | confidence_range=(1619-2459)
pagi_cctv_006_20260518_071450.mp4   pred=2153 veh/hr | confidence_range=(1709-2596)
pagi_cctv_009_20260518_071625.mp4   pred=2526 veh/hr | confidence_range=(2005-3046)
pagi_cctv_014_20260518_072255.mp4   pred=2105 veh/hr | confidence_range=(1671-2539)
pagi_cctv_015_20260518_072703.mp4   pred=1963 veh/hr | confidence_range=(1558-2367)
siang_cctv_001_20260518_163906.mp4  pred=1950 veh/hr | confidence_range=(1548-2351)
siang_cctv_003_20260518_165921.mp4  pred=1966 veh/hr | confidence_range=(1561-2371)
siang_cctv_004_20260518_170659.mp4  pred=2778 veh/hr | confidence_range=(2206-3350)
siang_cctv_005_20260518_171425.mp4  pred=2077 veh/hr | confidence_range=(1649-2506)
siang_cctv_007_20260518_160452.mp4  pred=

## 26. LSTM Validation for JSON

In [ ]:
summary = {
    "total_videos_processed": len(videos),
    "total_videos_failed": len(failed_videos),
    "failed_videos": failed_videos,

    "seq_len_used": SEQ_LEN,
    "horizon": HORIZON,
    "model_mape_15m": float(MODEL_MAPE),
    "feature_count": NUM_FEATURES,

    "runtime_config": {
        "DEMO_MODE": DEMO_MODE,
        "MAX_VIDEOS_PER_PERIOD": MAX_VIDEOS_PER_PERIOD,
        "MAX_FRAMES_PER_VIDEO": MAX_FRAMES_PER_VIDEO,
        "FRAME_STRIDE": FRAME_STRIDE,
        "YOLO_IMGSZ": YOLO_IMGSZ,
        "YOLO_CONF": YOLO_CONF,
        "SAVE_DETECTED_VIDEO": SAVE_DETECTED_VIDEO,
        "SAVE_EVERY_N_FRAMES": SAVE_EVERY_N_FRAMES,
    },

    "weather_sources": list(set(r.get("_weather_source", "") for r in video_feature_rows)),

    "videos": videos,

    # LSTM MODEL SUMMARY
    "lstm_validation": {
        "model_type": "LSTM",
        "model_mape_15m": float(MODEL_MAPE),

        "prediction_distribution": {
            "min_prediction": pred_min,
            "max_prediction": pred_max,
            "mean_prediction": pred_mean,
            "std_deviation": pred_std,
            "prediction_spread": pred_spread,
            "relative_spread_percent": relative_spread * 100
        },

        "global_confidence_range": {
            "lower_bound_mean": float(pred_mean * (1 - MODEL_MAPE / 100)),
            "upper_bound_mean": float(pred_mean * (1 + MODEL_MAPE / 100))
        },

        "correlation_vehicle_count_prediction": corr_value,

        "analysis": {
            "prediction_variability": (
                "Low" if relative_spread < 0.10 else
                "Moderate" if relative_spread < 0.20 else
                "High"
            ),

            "output_collapse_detected": bool(relative_spread < 0.02),

            "correlation_strength": (
                "Undefined" if corr_value is None else
                "Weak" if abs(corr_value) < 0.20 else
                "Moderate" if abs(corr_value) < 0.50 else
                "Strong"
            ),

            "interpretation": (
                "Model relies more on temporal patterns than vehicle count"
                if corr_value is not None and abs(corr_value) < 0.2 else
                "Vehicle count contributes significantly"
                if corr_value is not None else
                "Correlation not available"
            )
        }
    }
}

## 27. Save Final JSON

In [ ]:
summary_path = os.path.join(OUTPUT_DIR, "inference_summary.json")

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Saved: {summary_path}")

Saved: /content/outputs/inference_summary.json


## 28. Zip Outputs and Download

In [ ]:
import shutil
zip_path = '/content/traffix_inference_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"Created: {zip_path}.zip")

from google.colab import files
files.download(f"{zip_path}.zip")

Created: /content/traffix_inference_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 29. Final Summary

In [ ]:
print("\n=== INFERENCE COMPLETE ===")
print(f"  Processed  : {len(prediction_records)} videos")
print(f"  Failed     : {len(failed_videos)}")
print(f"  LSTM MAPE  : {MODEL_MAPE}%")
print()
for r in prediction_records:
    print(f"  {r['camera_id']} | {r['source_video']}")
    print(f"    {r['date']} {r['time']} | {r['period']}")
    if r['latitude']:
        print(f"    lat={r['latitude']}  lon={r['longitude']}")
    print(f"    weather={r['weather']} {r['weather_temp_c']}C")
    print(f"    vc={r['vehicle_count']} veh/min | vol={r['volume_veh_per_hour']} veh/hr")
    print(f"    speed={r['avg_speed_kmh']} km/h | density={r['density_percent']}%")
    print(f"    congestion={r['congestion_level']} | pred_15m={r['predicted_volume_15m']}")
    print(f"    {r['ai_insight']}")
    print()


=== INFERENCE COMPLETE ===
  Processed  : 21 videos
  Failed     : 0
  LSTM MAPE  : 20.6046%

  CAM_001 | pagi_cctv_001_20260518_070935.mp4
    5/18/2026 6:09:33 | Morning Rush
    lat=-6.2142883  lon=106.6820298
    weather=Cloudy 25.6C
    vc=6 veh/min | vol=1140 veh/hr
    speed=49.02 km/h | density=5.49%
    congestion=Low | pred_15m=2027.3
    Stable traffic with low density (5.5%) and speed 49.0 km/h. Peak-hour (Morning Rush) increases traffic demand.

  CAM_002 | pagi_cctv_002_20260518_070954.mp4
    5/18/2026 6:15:26 | Morning Rush
    lat=-6.2142883  lon=106.6820298
    weather=Cloudy 26.8C
    vc=33 veh/min | vol=2100 veh/hr
    speed=45.07 km/h | density=28.53%
    congestion=Low | pred_15m=2025.24
    Stable traffic with low density (28.5%) and speed 45.1 km/h. Peak-hour (Morning Rush) increases traffic demand.

  CAM_003 | pagi_cctv_004_20260518_071356.mp4
    5/18/2026 6:18:01 | Morning Rush
    lat=-6.2142883  lon=106.6820298
    weather=Cloudy 25.6C
    vc=19 veh/min |